# 07h — Multi-source evidence triangulation

Combines already-computed stable-secondary evidence from:

- Stage 05c behavioral independent evidence;
- Stage 07e historical OSM context;
- Stage 07g BCL POI 2008 lexical context.

This stage does **not** create a WORK/OFFICE score. It preserves source-native metrics, validates that all sources refer to the same candidate location, and summarizes convergence / disagreement patterns across the frozen 9-user stable-secondary cohort.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import pandas as pd

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/root/geolife'))

def run(cmd,cwd=None):
    subprocess.run(cmd,cwd=cwd,check=True)

if not (REPO_DIR/'.git').exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)])
else:
    run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    run(['git','-C',str(REPO_DIR),'checkout',BRANCH])
    run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'])

run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR)
for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path: sys.path.insert(0,p)

spec=importlib.util.spec_from_file_location('s07h',REPO_DIR/'analysis'/'07h_evidence_triangulation.py')
assert spec is not None and spec.loader is not None
s07h=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=s07h
spec.loader.exec_module(s07h)

print('sha:',subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())
print(s07h.synthetic_self_check())


In [ ]:
V=Path('/mnt/geolife-data')
C=V/'cache'/'cp2_v2'/'07h_evidence_triangulation'; C.mkdir(parents=True,exist_ok=True)

WORK=V/'cache'/'cp2_v2'/'05b_home_consensus_adaptive_work'/'adaptive_work_patterns_42_private.pkl'
BEHAVIOR=V/'cache'/'cp2_v2'/'05c_stable_secondary_evidence'/'candidate_comparison_private.pkl'
OSM=V/'cache'/'cp2_v2'/'07e_semantic_distance_alignment'/'stable_secondary_user_comparisons_private.pkl'
BCL=V/'cache'/'cp2_v2'/'07g_bcl_poi_2008_alignment'/'stable_secondary_bcl_comparisons_private.pkl'

for required in [WORK,BEHAVIOR,OSM,BCL]:
    if not required.exists(): raise FileNotFoundError(f'Run upstream stage first: {required}')

work=pd.read_pickle(WORK)
behavior=pd.read_pickle(BEHAVIOR)
osm=pd.read_pickle(OSM)
bcl=pd.read_pickle(BCL)

print('05b stable-secondary rows:',int(work['window_pattern'].eq('stable_secondary_anchor').sum()))
print('05c fair-comparator users:',behavior['user_id'].astype(str).nunique())
print('07e OSM comparison users:',osm['user_id'].astype(str).nunique())
print('07g BCL comparison users:',bcl['user_id'].astype(str).nunique())

assert int(work['window_pattern'].eq('stable_secondary_anchor').sum())==9
assert behavior['user_id'].astype(str).nunique()==7
assert osm['user_id'].astype(str).nunique()==9
assert bcl['user_id'].astype(str).nunique()==9


In [ ]:
panel=s07h.build_user_evidence_panel(work,behavior,osm,bcl)
coverage=s07h.summarize_source_coverage(panel)
direction=s07h.summarize_external_direction(panel)
context_overlap=s07h.summarize_candidate_context_overlap(panel,threshold_m=100.0)
signatures=s07h.summarize_triangulation_signatures(panel)
behavior_external=s07h.summarize_behavior_external_relationship(panel)
snapshot=s07h.decision_snapshot(panel)

assert len(panel)==9
assert panel['candidate_location_id'].notna().all()
print('candidate identity validation: PASS across available 05c / 07e / 07g evidence')


In [ ]:
print('SOURCE COVERAGE')
display(coverage)

print('EXTERNAL SOURCE DIRECTION AGREEMENT BY RADIUS')
display(direction)

print('CANDIDATE CONTEXT OVERLAP AT 100m')
display(context_overlap)

print('TRIANGULATION SIGNATURES AT 100m')
display(signatures)

print('BEHAVIOR BAND × NUMBER OF POSITIVE EXTERNAL SOURCES AT 100m')
display(behavior_external)

print('DECISION SNAPSHOT')
display(snapshot)


In [ ]:
# Private user-level panel
panel.to_pickle(C/'stable_secondary_triangulation_private.pkl')

# Aggregate outputs only
coverage.to_csv(C/'source_coverage.csv',index=False)
direction.to_csv(C/'external_direction_by_threshold.csv',index=False)
context_overlap.to_csv(C/'candidate_context_overlap_100m.csv',index=False)
signatures.to_csv(C/'triangulation_signature_summary.csv',index=False)
behavior_external.to_csv(C/'behavior_external_relationship.csv',index=False)
snapshot.to_csv(C/'decision_snapshot.csv',index=False)
print('saved:',C)


## Interpretation boundary

- `behavior_primary_support` is the existing Stage-05c criterion: at least 3 of 4 independent behavioral axes are top-1 within user.
- `behavior_directional_majority` is weaker/descriptive: candidate beats the peer median on at least 3 of 4 axes.
- OSM/BCL positive or negative signs mean only candidate-vs-peer context differences at the same radius.
- `strict_three_way_convergence_100m` is an audit flag, not a semantic class.
- No combination of flags may be emitted as HOME, WORK, OFFICE, occupation, employer, or employment status.
